# Real Estate Feasibility Engine – Python App
## Step 5: Automatic test suite – one click checks everything

**Why this matters:** from now on we will keep changing code (app screens, stress tests, exports). A test suite is a list of
small automatic checks. One click re-runs **all of them in a few seconds** and tells you instantly if anything broke.

**What gets tested (141 checks)**
| File | What it protects |
|---|---|
| `test_1_inputs.py` | the sample is valid, save/load works, the input checks catch 9 kinds of mistakes, project-type filter |
| `test_2_timeline_curves.py` | all 5 curve shapes total 100% for 8 window lengths, calendar & financial year, product quarters = Excel, links, fallback, scenarios |
| `test_3_collections.py` | every rupee booked is collected, Villa Q5 worked example (₹21.39 Cr), JV split, milestone quarters, two independent methods agree in 8 situations |
| `test_4_cashflow.py` | P&L and key metrics = Excel ('golden values'), Worst Case & Outright = Excel, IRR/NPV maths, and **accounting identities in 54 situations** |
| `test_5_excel.py` | full line-by-line comparison with your Excel FINAL file (skipped if the file is not in `data/`) |

**Before you start:** Steps 1–4 must have been run once. Run cells top to bottom with **Shift + Enter**.

### Cell 1 – Connect to Google Drive and open the project folder

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/feasibility_app'
except ImportError:
    BASE = os.path.abspath('feasibility_app')
os.chdir(BASE)
for f in ['inputs.py', 'timeline.py', 'curves.py', 'collections.py', 'costs.py', 'cashflow.py', 'excel_check.py']:
    assert os.path.exists(f'engine/{f}'), f'engine/{f} missing – please run the earlier steps first.'
os.makedirs('tests', exist_ok=True)
print('Working in:', BASE)
print('Excel FINAL file in data/:', os.path.exists('data/Feasibility_Engine_FINAL.xlsx'), '(if False, the Excel tests are skipped)')

### Cells 2–7 – Write the test files into `feasibility_app/tests/`
Each test is a short function whose name says what it checks, e.g. `test_collections_equal_revenue_for_every_product`.
**Golden values** are the Excel results for the sample project – if the Python ever drifts from Excel, a test fails.

In [ ]:
%%writefile tests/conftest.py
"""Shared set-up for all tests: makes the 'engine' package importable and provides ready-made results."""
import copy, os, sys

import pytest

BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))   # the feasibility_app folder
sys.path.insert(0, BASE)

from engine.inputs import sample_project          # noqa: E402
from engine.cashflow import run_model             # noqa: E402

EXCEL_PATH = os.path.join(BASE, "data", "Feasibility_Engine_FINAL.xlsx")


@pytest.fixture
def project():
    """A fresh copy of the sample project for every test (tests can change it freely)."""
    return sample_project()


@pytest.fixture(scope="session")
def base():
    """The full model results for the sample project (calculated once)."""
    return run_model(sample_project())


def variant(change):
    """Sample project with one change applied, e.g. variant(lambda p: setattr(p.info, 'scenario', 'Worst Case'))."""
    p = sample_project()
    change(p)
    return p


In [ ]:
%%writefile tests/test_1_inputs.py
"""STEP 1 – input structure: the sample is valid, saving works, and the checks catch mistakes."""
import copy
from datetime import date

from engine.inputs import CRORE, load_project, save_project, to_dict, validate


def test_sample_project_has_no_input_errors(project):
    assert validate(project) == []


def test_sample_matches_excel_inputs(project):
    act = project.active_products()
    assert len(act) == 4
    assert sum(p.area_sqft for p in act) == 920_000
    assert sum(p.area_sqft * p.rate for p in act) / CRORE == 803.0
    assert abs(sum(p.area_sqft * p.construction_rate for p in act) / CRORE - 270.4) < 1e-9


def test_save_and_load_gives_identical_project(project, tmp_path):
    path = tmp_path / "p.json"
    save_project(project, str(path))
    assert to_dict(load_project(str(path))) == to_dict(project)


def _errors_after(project, change):
    p = copy.deepcopy(project)
    change(p)
    return " | ".join(validate(p))


def test_catches_non_quarter_end_start_date(project):
    assert "quarter-end" in _errors_after(project, lambda p: setattr(p.info, "start_date", date(2027, 7, 15)))


def test_catches_payment_plan_not_100_percent(project):
    assert "must be exactly 100%" in _errors_after(project, lambda p: setattr(p.plan("CLP – Apartment").milestones[2], "pct", 0.05))


def test_catches_link_to_product_below(project):
    assert "ABOVE" in _errors_after(project, lambda p: setattr(p.products[1].timing, "launch_link_product", "Studios"))


def test_catches_booking_lag_above_12(project):
    assert "booking lag" in _errors_after(project, lambda p: setattr(p.plan("CLP – Apartment").milestones[1], "value", 13))


def test_catches_zone_shares_above_100(project):
    assert "Zone shares" in _errors_after(project, lambda p: setattr(p.zones[0], "share", 0.9))


def test_catches_advance_instalments_too_big(project):
    assert "exceed the total advance" in _errors_after(project, lambda p: setattr(p.initial, "dd_amount", 50))


def test_catches_invalid_bar_combination(project):
    assert "No BAR %" in _errors_after(project, lambda p: setattr(p.bar, "land_use", "Industrial"))


def test_catches_no_active_product(project):
    def change(p):
        p.info.project_type = "Plotted"
    assert "No product is active" in _errors_after(project, change)


def test_catches_duplicate_product_names(project):
    assert "different name" in _errors_after(project, lambda p: setattr(p.products[2], "name", "Villas"))


def test_project_type_filters_products(project):
    expected = {"Villa": ["Villas"],
                "Apartments": ["Apartments – Towers A & B", "Studios", "High-street retail"],
                "Mixed Development": ["Villas", "Apartments – Towers A & B", "Studios", "High-street retail"]}
    for ptype, names in expected.items():
        p = copy.deepcopy(project)
        p.info.project_type = ptype
        assert [x.name for x in p.active_products()] == names


In [ ]:
%%writefile tests/test_2_timeline_curves.py
"""STEP 2 – timeline and curves: shapes always total 100%, quarters match Excel, links and scenarios work."""
import math

import numpy as np
import pytest

from engine.curves import build_curves, curve_weights, spread
from engine.timeline import build_timeline, quarter_end, fy_label
from datetime import date
from conftest import variant

CURVES = ["Upfront", "Even", "Front-loaded", "Back-loaded", "S-curve"]


@pytest.mark.parametrize("name", CURVES)
@pytest.mark.parametrize("n", [1, 2, 3, 7, 12, 16, 24, 40])
def test_every_curve_totals_exactly_100_percent(name, n):
    w = curve_weights(name, n)
    assert len(w) == n and abs(w.sum() - 1) < 1e-12 and (w >= 0).all()


def test_s_curve_is_symmetric_and_peaks_in_middle():
    w = curve_weights("S-curve", 12)
    assert np.allclose(w, w[::-1]) and w.argmax() in (5, 6)


def test_front_and_back_loaded_mirror_each_other():
    assert np.allclose(curve_weights("Front-loaded", 9), curve_weights("Back-loaded", 9)[::-1])


def test_spread_places_curve_in_window():
    s = spread("Even", 5, 8)
    assert s[:4].sum() == 0 and abs(s[4:8].sum() - 1) < 1e-12 and s[8:].sum() == 0


def test_calendar_and_financial_year():
    assert quarter_end(date(2027, 6, 30), 1) == date(2027, 6, 30)
    assert quarter_end(date(2027, 6, 30), 3) == date(2027, 12, 31)
    assert quarter_end(date(2027, 6, 30), 4) == date(2028, 3, 31)
    assert fy_label(date(2027, 6, 30)) == "FY 2027-28" and fy_label(date(2028, 3, 31)) == "FY 2027-28"


EXCEL_QUARTERS = {"Villas": (3, 10, 4, 15), "Apartments – Towers A & B": (4, 15, 5, 20),
                  "Studios": (6, 13, 6, 17), "High-street retail": (8, 15, 6, 17)}


def test_product_quarters_match_excel(base):
    tl = base["timeline"]
    got = {n: (t.launch, t.sales_end, t.cons_start, t.cons_end) for n, t in tl.products.items()}
    assert got == EXCEL_QUARTERS and tl.last_q == 20


def test_linked_products_follow_the_lead_product():
    tl = build_timeline(variant(lambda p: setattr(p.products[0], "launch_q", 5)))
    assert tl.products["Villas"].launch == 5
    assert tl.products["Apartments – Towers A & B"].launch == 6        # 1 quarter after villas
    assert tl.products["High-street retail"].launch == 10              # 4 quarters after apartments


def test_fallback_to_fixed_quarter_when_linked_product_inactive():
    tl = build_timeline(variant(lambda p: setattr(p.info, "project_type", "Apartments")))
    t = tl.products["Apartments – Towers A & B"]
    assert t.launch == 4 and "fallback" in t.note


def test_worst_case_delays_and_slows_sales():
    tl = build_timeline(variant(lambda p: setattr(p.info, "scenario", "Worst Case")))
    v = tl.products["Villas"]
    assert (v.launch, v.sales_end, v.cons_start) == (5, 16, 6)          # +2 delay, 8 x 1.5 = 12 quarters


def test_longer_construction_extends_project():
    def change(p): p.products[1].construction_duration_q = 24
    assert build_timeline(variant(change)).last_q == 28


def test_all_curve_rows_total_100_percent(base):
    cv = base["curves"]
    for arr in list(cv.sales_pct.values()) + list(cv.cons_pct.values()):
        assert abs(arr.sum() - 1) < 1e-9
    for k, arr in cv.cost_pct.items():
        assert arr is not None and (abs(arr.sum() - 1) < 1e-9 or arr.sum() == 0)
    assert abs(cv.cons_cum["Villas"][-1] - 1) < 1e-9


In [ ]:
%%writefile tests/test_3_collections.py
"""STEP 3 – collections: every rupee booked is collected, two methods agree, worked example holds."""
import numpy as np
import pytest

from engine.cashflow import run_model
from engine.collections import verify_by_cohorts
from conftest import variant


def test_collections_equal_revenue_for_every_product(base):
    co, cv = base["collections"], base["curves"]
    for name, cash in co.collections.items():
        assert abs(cash.sum() - cv.revenue[name]) < 1e-6
        assert co.receivables[name].min() > -1e-6                # never paid before booked
        assert abs(co.receivables[name][-1]) < 1e-6              # fully cleared


def test_worked_example_villa_q5_is_21_39_crore(base):
    assert abs(base["collections"].collections["Villas"][4] - 21.388888888888889) < 1e-9


def test_jv_split_32_68(base):
    co = base["collections"]
    assert abs(co.landowner.sum() - 256.96) < 1e-6 and abs(co.developer.sum() - 546.04) < 1e-6


def test_villa_milestones_fall_due_in_excel_quarters(base):
    due = {s.name: s.due_q for s in base["collections"].schedule if s.product == "Villas" and s.due_q}
    assert due == {"Commencement of construction": 5, "Ground plinth": 7, "First floor roof casting": 8,
                   "Second floor roof casting": 10, "Third floor roof casting": 11,
                   "Completion certificate (CC)": 14, "Possession": 15}


SCENARIOS = {
    "base": lambda p: None,
    "worst": lambda p: setattr(p.info, "scenario", "Worst Case"),
    "best": lambda p: setattr(p.info, "scenario", "Best Case"),
    "lag0": lambda p: setattr(p, "catch_up_lag", 0),
    "lag2": lambda p: setattr(p, "catch_up_lag", 2),
    "tower17": lambda p: setattr(p.products[1], "clp_plan", "CLP – 17-Floor Tower"),
    "apartments": lambda p: setattr(p.info, "project_type", "Apartments"),
    "longer_build": lambda p: setattr(p.products[1], "construction_duration_q", 24),
}


@pytest.mark.parametrize("name", SCENARIOS)
def test_two_independent_methods_agree(name):
    r = run_model(variant(SCENARIOS[name]))
    assert r["problems"] == []
    assert verify_by_cohorts(variant(SCENARIOS[name]), r["timeline"], r["curves"], r["collections"]) < 1e-9
    assert abs(r["collections"].total.sum() - sum(r["curves"].revenue.values())) < 1e-6


In [ ]:
%%writefile tests/test_4_cashflow.py
"""STEP 4 – costs, cash flow, P&L and returns: golden values (verified against Excel) and accounting identities."""
import itertools

import numpy as np
import pytest

from engine.cashflow import irr, npv, run_model
from conftest import variant

# golden values = the Excel FINAL model's results for the sample project
GOLDEN = dict(revenue=803.0, total_cost=630.6053209296, ebitda=172.3946790704, pat=127.572062512096,
              developer_irr=1.1017637673975016, project_irr=4.490612021677777, developer_npv=90.62811362965084,
              peak_investment=34.2873209296, equity_multiple=1.4732968016982388, roce=1.2009637125967791,
              cagr=0.4033845607907973, initial_investment=28.7093901776)


def test_pnl_matches_excel(base):
    pnl = base["cashflow"].pnl
    for k in ("revenue", "total_cost", "ebitda", "pat"):
        assert abs(pnl[k] - GOLDEN[k]) < 1e-6, k


def test_key_metrics_match_excel(base):
    m = base["cashflow"].metrics
    for k in ("developer_irr", "project_irr", "developer_npv", "peak_investment", "equity_multiple", "roce", "cagr", "initial_investment"):
        assert abs(m[k] - GOLDEN[k]) < 1e-6, k
    assert (m["peak_quarter"], m["payback_q"], m["breakeven_project_q"], m["breakeven_developer_q"]) == (2, 8, 3, 3)


def test_bar_and_land_amounts(base):
    a = base["costs"].amounts
    assert abs(a["bar"] - 3.154574336) < 1e-6 and abs(a["land"] - 256.96) < 1e-6 and abs(a["brok"] - 2.124) < 1e-9


@pytest.mark.parametrize("name, change, irr_, peak, payback", [
    ("worst", lambda p: setattr(p.info, "scenario", "Worst Case"), 0.40241041469118866, 36.3601955115398, 18),
    ("outright_llp", lambda p: (setattr(p.info, "deal_type", "Outright"), setattr(p.land, "payment_curve", "Even"),
                                setattr(p.land, "payment_end_q", 4), setattr(p.info, "entity", "LLP")),
     0.3561027999778046, 203.09589048611082, 12),
])
def test_other_scenarios_match_excel(name, change, irr_, peak, payback):
    m = run_model(variant(change))["cashflow"].metrics
    assert abs(m["developer_irr"] - irr_) < 1e-6 and abs(m["peak_investment"] - peak) < 1e-6 and m["payback_q"] == payback


def test_irr_and_npv_helpers_on_known_numbers():
    assert abs(irr(np.array([-100.0, 110.0])) - 0.10) < 1e-9
    assert abs(irr(np.array([-100.0, 0.0, 121.0])) - 0.10) < 1e-9
    assert irr(np.array([100.0, 50.0])) is None                        # no investment -> no IRR
    assert abs(npv(0.10, np.array([110.0])) - 100.0) < 1e-9


# accounting identities that must hold in EVERY situation
SITUATIONS = list(itertools.product(["Base Case", "Best Case", "Worst Case"], ["JV", "Outright"],
                                    ["Mixed Development", "Villa", "Apartments"], [0, 1, 2]))


@pytest.mark.parametrize("scenario, deal, ptype, lag", SITUATIONS)
def test_accounting_identities_hold(scenario, deal, ptype, lag):
    def change(p):
        p.info.scenario, p.info.deal_type, p.info.project_type, p.catch_up_lag = scenario, deal, ptype, lag
    p = variant(change)
    r = run_model(p)
    cf, cr, co = r["cashflow"], r["costs"], r["collections"]
    assert r["problems"] == []
    assert abs(cf.developer_cf.sum() - cf.pnl["ebitda"]) < 1e-6                     # cash = profit
    for c in p.costs:                                                                # each cost paid exactly once
        expected = 0.0 if (deal == "JV" and c.key == "land") else cr.amounts[c.key]
        assert abs(cf.outflows[c.key].sum() - expected) < 1e-6, c.name
    assert abs(cf.lo_paid.sum() - co.landowner.sum()) < 1e-6                         # landowner paid his share
    assert abs(cf.pnl["revenue"] - cf.pnl["total_cost"] - cf.pnl["ebitda"]) < 1e-9
    assert cf.metrics["peak_investment"] >= 0


In [ ]:
%%writefile tests/test_5_excel.py
"""Full comparison with the Excel FINAL file (skipped automatically if the file is not in data/)."""
import os

import numpy as np
import pytest

from conftest import EXCEL_PATH

pytestmark = pytest.mark.skipif(not os.path.exists(EXCEL_PATH), reason="Excel FINAL file not found in data/")


@pytest.fixture(scope="module")
def excel():
    from engine.excel_check import read_excel_cashflow, read_excel_collections, read_excel_curves, read_excel_timeline
    return dict(timeline=read_excel_timeline(EXCEL_PATH), curves=read_excel_curves(EXCEL_PATH),
                collections=read_excel_collections(EXCEL_PATH), cashflow=read_excel_cashflow(EXCEL_PATH))


def test_excel_product_quarters(base, excel):
    got = {n: (t.launch, t.sales_end, t.cons_start, t.cons_end) for n, t in base["timeline"].products.items()}
    assert got == excel["timeline"]


def test_excel_curves(base, excel):
    cv, x = base["curves"], excel["curves"]
    for n in cv.sales_pct:
        assert np.abs(cv.sales_pct[n] - x["sales"][n]).max() < 1e-9
        assert np.abs(cv.cons_pct[n] - x["cons"][n]).max() < 1e-9


def test_excel_cost_phasing(base, excel):
    x = excel["curves"]
    for i, c in enumerate(_project_costs()):
        assert np.abs(base["curves"].cost_pct[c.key] - x["cost"][x["cost_order"][i]]).max() < 1e-9, c.name


def test_excel_collections(base, excel):
    co, x = base["collections"], excel["collections"]
    for n in co.collections:
        assert np.abs(co.collections[n] - x["collections"][n]).max() < 1e-9
    assert np.abs(co.total - x["total"]).max() < 1e-9 and np.abs(co.landowner - x["landowner"]).max() < 1e-9


def test_excel_cash_rows_costs_and_pnl(base, excel):
    cf, cr, x = base["cashflow"], base["costs"], excel["cashflow"]
    assert max(abs(a - b) for a, b in zip(cr.amounts.values(), x["costs"])) < 1e-9
    for o, xo in zip(cf.outflows.values(), x["outflows"]):
        assert np.abs(o - xo).max() < 1e-9
    for k, v in x["rows"].items():
        assert np.abs(getattr(cf, k) - v).max() < 1e-9, k
    for k, v in x["pnl"].items():
        assert abs(cf.pnl[k] - v) < 1e-9, k


def test_excel_all_18_key_metrics(base, excel):
    keys = ["project_irr", "developer_irr", "equity_multiple", "roce", "cagr", "pat_margin", "ebitda_margin", "peak_investment",
            "land_cost_to_gdv", "initial_investment_to_gdv", "breakeven_project_q", "breakeven_developer_q", "payback_q",
            "project_npv", "developer_npv", "developer_irr_post_tax", "largest_quarter_deficit", "cash_multiple"]
    for (label, xv), k in zip(excel["cashflow"]["metrics"].items(), keys):
        pv = base["cashflow"].metrics[k]
        if isinstance(xv, str):
            assert xv == f"Q{pv}", label
        else:
            assert abs(pv - xv) < 1e-6, label


def _project_costs():
    from engine.inputs import sample_project
    return sample_project().costs


### Cell 8 – ▶ THE ONE-CLICK TEST RUN
Run this cell **any time you change code**. Green ✅ = everything still works.

In [ ]:
import subprocess, sys, xml.etree.ElementTree as ET, pandas as pd

def run_tests(folder='tests', show_details=False):
    res = subprocess.run([sys.executable, '-m', 'pytest', folder, '-q', '-rs', '--tb=short', '--junitxml=test_report.xml'],
                         capture_output=True, text=True)
    rows = {}
    for tc in ET.parse('test_report.xml').getroot().iter('testcase'):
        f = tc.get('classname', '').split('.')[-1] or tc.get('file', '')
        status = 'failed' if tc.find('failure') is not None or tc.find('error') is not None else 'skipped' if tc.find('skipped') is not None else 'passed'
        rows.setdefault(f, {'passed': 0, 'failed': 0, 'skipped': 0})[status] += 1
    table = pd.DataFrame(rows).T
    table.loc['TOTAL'] = table.sum()
    display(table)
    if res.returncode == 0:
        print(f"✅ ALL {int(table.loc['TOTAL','passed'])} TESTS PASSED" + (f" ({int(table.loc['TOTAL','skipped'])} skipped – Excel file not in data/)" if table.loc['TOTAL','skipped'] else ''))
    else:
        print(f"❌ {int(table.loc['TOTAL','failed'])} TEST(S) FAILED – details below:\n")
        print(res.stdout[-6000:])
    if show_details: print(res.stdout)
    return res.returncode == 0

run_tests()

### Cell 9 – What a failure looks like (a safe demonstration)
We copy the engine to a temporary folder, plant **one tiny bug** (the S-curve made 1% too big), and run the same tests on the copy.
Your real files are not touched.

In [ ]:
import shutil, tempfile
demo = os.path.join(tempfile.mkdtemp(), 'feasibility_app')
shutil.copytree('engine', os.path.join(demo, 'engine')); shutil.copytree('tests', os.path.join(demo, 'tests'))
path = os.path.join(demo, 'engine', 'curves.py')
code_text = open(path, encoding='utf-8').read().replace('* math.sin(math.pi / (2 * n))', '* math.sin(math.pi / (2 * n)) * 1.01')
open(path, 'w', encoding='utf-8').write(code_text)
print('Bug planted in the COPY: S-curve weights are now 1% too large.\n')
res = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--tb=no'], cwd=demo, capture_output=True, text=True)
failed = [l.split('::')[-1].split(' ')[0] for l in res.stdout.splitlines() if l.startswith('FAILED')]
print(res.stdout.strip().splitlines()[-1])
print('\nA few of the tests that caught it:')
for name in sorted(set(f.split('[')[0] for f in failed))[:8]: print('  ✖', name)
shutil.rmtree(os.path.dirname(demo))
print('\nTemporary copy deleted – your real engine is unchanged. Re-run Cell 8 to confirm ✅.')

### Cell 10 – Optional: see the name of every single test
Useful when you want to know exactly what is covered.

In [ ]:
res = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--collect-only'], capture_output=True, text=True)
names = [l for l in res.stdout.splitlines() if '::' in l]
print(len(names), 'tests:'); print('\n'.join(names[:200]))

---
### ✅ Step 5 complete – your safety net is in place
**The habit from now on:** after any change to the code → open this notebook → run **Cell 1** and **Cell 8** → look for ✅.

`feasibility_app/` now contains:
* `engine/` – the complete calculation engine (Steps 1–4)
* `tests/` – 141 automatic checks
* `data/` – the sample project file (and your Excel FINAL file for the comparison tests)

**Next – Step 6:** the app's input screens with **Streamlit** – forms and editable tables instead of Excel cells, with the checks shown right next to each field.